<a href="https://colab.research.google.com/github/Muneebshah1192/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

I will use supervised classification models to predict whether a content item is showing a declining trend.

The target is derived from the historical `trend_direction` field, where a value of `down` represents a declining item. The target-related fields `trend_direction` and `trend_pct` will not be used as input features because doing so would leak information about the outcome into the model.

I will first use Logistic Regression as a simple and interpretable model. I will then test Random Forest as a more flexible nonlinear model.

These models fit the lane because the business question is about prioritizing content that may need attention. The ML-07 baseline provides a simple ranking based on search volume, freshness, and CTR. ML-08 tests whether learned patterns from the available historical features can provide useful decision-support beyond that heuristic.

Model complexity will not be treated as automatically better. The main comparison will use the same evaluation concept and held-out client groups so that the result reflects performance on unseen clients.

In [6]:
import numpy as np
import pandas as pd

df = df.copy()

# Target: 1 = declining, 0 = not declining
df["declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

print("Target distribution:")
print(df["declining_label"].value_counts())

print("\nTarget proportions:")
print(df["declining_label"].value_counts(normalize=True).round(3))

Target distribution:
declining_label
1    16262
0    13738
Name: count, dtype: int64

Target proportions:
declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
!find /content -name "content_refresh_anonymized.csv" -type f

In [3]:
!git clone https://github.com/Muneebshah1192/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 135 (delta 42), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 1.92 MiB | 14.04 MiB/s, done.
Resolving deltas: 100% (42/42), done.


In [4]:
!ls -lh /content/flyrank-ml-internship/data/raw/

total 6.5M
-rw-r--r-- 1 root root 6.5M Aug 11 10:54 content_refresh_anonymized.csv


In [5]:
import pandas as pd

DATA_PATH = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,NaN,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,NaN,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,91-180,4,20,0-30,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,NaN,NaN,11751,58,87,78,75,1,0,3,88,51,3626,22,35,4206,17,26,463,365+,6,22,0-30,NaN,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,NaN,gemini-3-flash-preview,19140,24,177,145,144,0,0,43,88,33,4211,10,14,6452,2,9,263,181-365,5,14,0-30,2000-3500,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


### Split design

I will use a client-grouped holdout rather than randomly splitting individual rows.

The dataset contains multiple content records associated with the same client. A random row-level split could place records from the same client in both training and evaluation data, making the evaluation more optimistic because the model could benefit from client-specific patterns.

Instead, complete clients are assigned to either the training or holdout set. This better represents the intended decision-support setting: evaluating whether the model can generalize to content from clients that were not used during training.

The holdout data will remain untouched during model fitting and comparison.

In [7]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"].astype(str)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        y=df["declining_label"],
        groups=groups
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Training rows:", len(train_df))
print("Holdout rows:", len(test_df))

print(
    "Training clients:",
    train_df["client_id"].nunique()
)

print(
    "Holdout clients:",
    test_df["client_id"].nunique()
)

overlap = set(train_df["client_id"]) & set(test_df["client_id"])

print("Client overlap:", len(overlap))

Training rows: 23837
Holdout rows: 6163
Training clients: 25
Holdout clients: 7
Client overlap: 0


In [8]:
#define features
target_col = "declining_label"

excluded_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    target_col
]

feature_candidates = [
    col
    for col in df.columns
    if col not in excluded_columns
]

print("Candidate features:")
print(feature_candidates)

Candidate features:
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']


In [9]:
#seperate the numerical and catagorical data
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X_train = train_df[feature_candidates]
X_test = test_df[feature_candidates]

y_train = train_df[target_col]
y_test = test_df[target_col]

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 29
Categorical features: 11


In [10]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [11]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

logistic_model.fit(X_train, y_train)

logistic_prob = logistic_model.predict_proba(
    X_test
)[:, 1]

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [12]:
#Trained Random Forest
from sklearn.ensemble import RandomForestClassifier

random_forest_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
                class_weight="balanced_subsample"
            )
        )
    ]
)

random_forest_model.fit(X_train, y_train)

random_forest_prob = random_forest_model.predict_proba(
    X_test
)[:, 1]

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [13]:
print(
    "Logistic probability range:",
    round(logistic_prob.min(), 4),
    "to",
    round(logistic_prob.max(), 4)
)

print(
    "Random Forest probability range:",
    round(random_forest_prob.min(), 4),
    "to",
    round(random_forest_prob.max(), 4)
)

Logistic probability range: 0.0 to 1.0
Random Forest probability range: 0.0 to 0.9867


In [14]:
#Created the Precision@50 evaluator
import numpy as np
import pandas as pd


def precision_at_k(y_true, scores, k=50):
    """
    Precision@K for a ranked list.
    """
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    top_indices = np.argsort(scores)[::-1][:k]

    return y_true[top_indices].mean()

In [16]:
from sklearn.preprocessing import MinMaxScaler

baseline_df = test_df.copy()

# Training-set medians only
sv_median = train_df["search_volume"].median()
freshness_median = train_df["days_since_last_update"].median()
ctr_median = train_df["ctr"].median()

# Create filled columns
baseline_df["search_volume"] = (
    baseline_df["search_volume"].fillna(sv_median)
)

baseline_df["days_since_last_update"] = (
    baseline_df["days_since_last_update"]
    .fillna(freshness_median)
)

baseline_df["ctr"] = (
    baseline_df["ctr"]
    .fillna(ctr_median)
)

# Fit scalers ONLY on training data
sv_scaler = MinMaxScaler()
freshness_scaler = MinMaxScaler()
ctr_scaler = MinMaxScaler()

sv_scaler.fit(
    train_df[["search_volume"]].fillna(sv_median)
)

freshness_scaler.fit(
    train_df[["days_since_last_update"]]
    .fillna(freshness_median)
)

ctr_scaler.fit(
    train_df[["ctr"]].fillna(ctr_median)
)

# Transform holdout using the training-fitted scalers
baseline_df["sv_norm"] = sv_scaler.transform(
    baseline_df[["search_volume"]]
)

baseline_df["freshness_norm"] = freshness_scaler.transform(
    baseline_df[["days_since_last_update"]]
)

baseline_df["ctr_norm"] = ctr_scaler.transform(
    baseline_df[["ctr"]]
)

# Reproduce ML-07 baseline logic
baseline_df["baseline_score"] = (
    0.4 * baseline_df["sv_norm"]
    + 0.4 * baseline_df["freshness_norm"]
    + 0.2 * (1 - baseline_df["ctr_norm"])
)

print(
    baseline_df["baseline_score"].describe()
)

count    6163.000000
mean        0.236716
std         0.038077
min         0.153770
25%         0.214309
50%         0.220430
75%         0.225641
max         0.430753
Name: baseline_score, dtype: float64


In [17]:
results = test_df[
    [
        "content_id",
        "client_id",
        "declining_label"
    ]
].copy()

results["baseline_score"] = baseline_df["baseline_score"].values
results["logistic_score"] = logistic_prob
results["random_forest_score"] = random_forest_prob

results.head()

,content_id,client_id,declining_label,baseline_score,logistic_score,random_forest_score
0,content_304f48230142,client_f369cb89fc,1,0.218964,0.890144,0.766667
1,content_a1fb4e703a9e,client_4e07408562,1,0.226193,0.999999,0.483333
5,content_d4084a4bc775,client_f369cb89fc,1,0.224262,0.859436,0.770000
13,content_a5a2fbc76336,client_8527a891e2,0,0.309731,0.612376,0.663333
19,content_af865035b328,client_f369cb89fc,1,0.216552,0.604700,0.646667


In [18]:
def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))
    top_indices = np.argsort(scores)[::-1][:k]

    return y_true[top_indices].mean()


baseline_p50 = precision_at_k(
    results["declining_label"],
    results["baseline_score"],
    50
)

logistic_p50 = precision_at_k(
    results["declining_label"],
    results["logistic_score"],
    50
)

random_forest_p50 = precision_at_k(
    results["declining_label"],
    results["random_forest_score"],
    50
)

print("ML-07 Baseline Precision@50:", round(baseline_p50, 4))
print("Logistic Regression Precision@50:", round(logistic_p50, 4))
print("Random Forest Precision@50:", round(random_forest_p50, 4))

ML-07 Baseline Precision@50: 0.44
Logistic Regression Precision@50: 1.0
Random Forest Precision@50: 1.0


In [19]:
comparison_table = pd.DataFrame({
    "Method": [
        "ML-07 Baseline",
        "Logistic Regression",
        "Random Forest"
    ],
    "Precision@50": [
        baseline_p50,
        logistic_p50,
        random_forest_p50
    ]
})

comparison_table["Improvement_vs_Baseline"] = (
    comparison_table["Precision@50"] - baseline_p50
)

comparison_table["Relative_Improvement_%"] = np.where(
    baseline_p50 != 0,
    (
        comparison_table["Improvement_vs_Baseline"]
        / baseline_p50
    ) * 100,
    np.nan
)

comparison_table.round(4)

,Method,Precision@50,Improvement_vs_Baseline,Relative_Improvement_%
0,ML-07 Baseline,0.44,0.00,0.0000
1,Logistic Regression,1.00,0.56,127.2727
2,Random Forest,1.00,0.56,127.2727


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and interpretation

The ML-07 baseline achieved a Precision@50 of 0.44 on the client-held-out evaluation set. Logistic Regression and Random Forest both achieved 1.00 Precision@50.

This means that all 50 of the highest-ranked pages from each ML model were observed to have a declining trend in this particular holdout set. The result is substantially higher than the baseline, which had 22 declining pages among its top 50.

The result should be treated as an observed holdout result rather than evidence of perfect generalization. In particular, Precision@50 evaluates only the top 50 ranked items, so it does not describe performance across the entire dataset.

The next step is to inspect the errors outside the top 50 and examine which features the models rely on. This helps determine whether the improvement reflects useful historical signals or a narrower pattern in the evaluation sample.

In [20]:
#Count errors across the whole holdout
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

results["logistic_pred"] = (
    results["logistic_score"] >= 0.5
).astype(int)

results["random_forest_pred"] = (
    results["random_forest_score"] >= 0.5
).astype(int)

print("Logistic Regression")
print(
    classification_report(
        results["declining_label"],
        results["logistic_pred"],
        digits=3
    )
)

print("\nRandom Forest")
print(
    classification_report(
        results["declining_label"],
        results["random_forest_pred"],
        digits=3
    )
)

Logistic Regression
              precision    recall  f1-score   support

           0      0.732     0.735     0.733      3014
           1      0.745     0.742     0.744      3149

    accuracy                          0.739      6163
   macro avg      0.738     0.739     0.738      6163
weighted avg      0.739     0.739     0.739      6163


Random Forest
              precision    recall  f1-score   support

           0      0.843     0.743     0.790      3014
           1      0.779     0.868     0.821      3149

    accuracy                          0.806      6163
   macro avg      0.811     0.805     0.805      6163
weighted avg      0.810     0.806     0.806      6163



In [21]:
#Confusion matrices
print("Logistic Regression confusion matrix:")
print(
    confusion_matrix(
        results["declining_label"],
        results["logistic_pred"]
    )
)

print("\nRandom Forest confusion matrix:")
print(
    confusion_matrix(
        results["declining_label"],
        results["random_forest_pred"]
    )
)

Logistic Regression confusion matrix:
[[2214  800]
 [ 811 2338]]

Random Forest confusion matrix:
[[2238  776]
 [ 417 2732]]


In [22]:
#Find the actual errors
logistic_errors = results[
    results["declining_label"]
    != results["logistic_pred"]
].copy()

random_forest_errors = results[
    results["declining_label"]
    != results["random_forest_pred"]
].copy()

print(
    "Logistic Regression errors:",
    len(logistic_errors)
)

print(
    "Random Forest errors:",
    len(random_forest_errors)
)

Logistic Regression errors: 1611
Random Forest errors: 1193


In [23]:
#Inspectd the errors
error_columns = [
    "content_id",
    "client_id",
    "declining_label",
    "logistic_score",
    "random_forest_score"
]

logistic_errors[
    error_columns
].head(20)

,content_id,client_id,declining_label,logistic_score,random_forest_score
13,content_a5a2fbc76336,client_8527a891e2,0,0.612376,0.663333
23,content_2da6ae9d0882,client_e629fa6598,1,0.392534,0.616667
25,content_033ae3e7aecf,client_f369cb89fc,1,0.446271,0.573333
36,content_bce275871a25,client_f369cb89fc,0,0.588161,0.700000
39,content_4595e8704e07,client_8527a891e2,1,0.326003,0.763333
43,content_1938955b34c4,client_f369cb89fc,1,0.202083,0.690000
47,content_40cb4af260c0,client_f369cb89fc,1,0.448451,0.736667
54,content_ff8ea1364b59,client_e629fa6598,1,0.482646,0.826667
58,content_caff51984338,client_e629fa6598,1,0.342773,0.816667
60,content_b9104a222d01,client_f369cb89fc,1,0.441418,0.606667


In [24]:
#Error summaryi
error_summary = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Total_holdout_rows": [
        len(results),
        len(results)
    ],
    "Errors": [
        len(logistic_errors),
        len(random_forest_errors)
    ],
    "Error_rate": [
        len(logistic_errors) / len(results),
        len(random_forest_errors) / len(results)
    ]
})

error_summary.round(4)

,Model,Total_holdout_rows,Errors,Error_rate
0,Logistic Regression,6163,1611,0.2614
1,Random Forest,6163,1193,0.1936


In [25]:
#Feature interpretation
fitted_preprocessor = logistic_model.named_steps["preprocess"]
fitted_lr = logistic_model.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()

coefficients = pd.DataFrame({
    "feature": feature_names,
    "coefficient": fitted_lr.coef_[0]
})

coefficients["abs_coefficient"] = (
    coefficients["coefficient"].abs()
)

top_lr_features = (
    coefficients
    .sort_values(
        "abs_coefficient",
        ascending=False
    )
    .head(15)
)

top_lr_features

,feature,coefficient,abs_coefficient
15,numeric__impressions_last_30d,-35.873439,35.873439
18,numeric__impressions_prev_30d,29.535083,29.535083
5,numeric__impressions_90d,1.921298,1.921298
70,categorical__position_tier_top_3,-1.253448,1.253448
19,numeric__clicks_prev_30d,1.058300,1.058300
16,numeric__clicks_last_30d,-0.985563,0.985563
37,categorical__main_intent_navigational,-0.547696,0.547696
8,numeric__sessions_90d,0.546320,0.546320
13,numeric__days_with_impressions,0.539104,0.539104
9,numeric__users_90d,-0.538767,0.538767


In [26]:
#Random Forest importance

fitted_rf = random_forest_model.named_steps["model"]

rf_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": fitted_rf.feature_importances_
})

top_rf_features = (
    rf_importance
    .sort_values(
        "importance",
        ascending=False
    )
    .head(15)
)

top_rf_features

,feature,importance
18,numeric__impressions_prev_30d,0.154649
15,numeric__impressions_last_30d,0.124397
5,numeric__impressions_90d,0.069190
25,numeric__avg_position,0.051128
13,numeric__days_with_impressions,0.050095
21,numeric__content_age_days,0.040644
17,numeric__sessions_last_30d,0.028415
3,numeric__word_count,0.026251
4,numeric__char_count,0.025859
24,numeric__ctr,0.025157


In [27]:
#Important leakage check
leakage_check = [
    col for col in feature_candidates
    if col in [
        "trend_direction",
        "trend_pct",
        "declining_label"
    ]
]

print("Leakage-prone columns in features:", leakage_check)

Leakage-prone columns in features: []


In [28]:
print("Holdout size:", len(results))

print(
    "Declining rate in holdout:",
    round(
        results["declining_label"].mean(),
        4
    )
)

print(
    "Declining pages in top 50 - baseline:",
    int(
        results.sort_values(
            "baseline_score",
            ascending=False
        )
        .head(50)["declining_label"]
        .sum()
    )
)

print(
    "Declining pages in top 50 - logistic:",
    int(
        results.sort_values(
            "logistic_score",
            ascending=False
        )
        .head(50)["declining_label"]
        .sum()
    )
)

print(
    "Declining pages in top 50 - random forest:",
    int(
        results.sort_values(
            "random_forest_score",
            ascending=False
        )
        .head(50)["declining_label"]
        .sum()
    )
)

Holdout size: 6163
Declining rate in holdout: 0.511
Declining pages in top 50 - baseline: 22
Declining pages in top 50 - logistic: 50
Declining pages in top 50 - random forest: 50


In [29]:
results[
    [
        "declining_label",
        "baseline_score",
        "logistic_score",
        "random_forest_score"
    ]
].head()

,declining_label,baseline_score,logistic_score,random_forest_score
0,1,0.218964,0.890144,0.766667
1,1,0.226193,0.999999,0.483333
5,1,0.224262,0.859436,0.770000
13,0,0.309731,0.612376,0.663333
19,1,0.216552,0.604700,0.646667


### Self-check

- [x] Every required section is filled with both explanation and supporting code.
- [x] The model uses a client-grouped holdout split.
- [x] There is no client overlap between training and holdout data.
- [x] `trend_direction`, `trend_pct`, and the derived target were excluded from model features.
- [x] The ML-07 baseline and learned models were evaluated on the same holdout data.
- [x] Precision@50 was used as the primary ranking metric.
- [x] Classification errors and feature interpretation were inspected.
- [x] Claims use careful language such as observed, measured, directional, and decision-support.
- [x] No client identifiers are displayed in the final analysis outputs.
- [ ] Notebook runs top-to-bottom without errors.
- [ ] Notebook is saved under `work/notebooks/w05_model.ipynb`.
- [ ] Changes are committed and pushed to the repository.

In [30]:
git add work/notebooks/w05_model.ipynb
git commit -m "Complete ML-08 modeling analysis"
git push

SyntaxError: invalid syntax (2276720793.py, line 1)

### ML-08 conclusion

The client-grouped holdout contained 6,163 records, with 51.1% labeled as declining.

The ML-07 baseline achieved Precision@50 of 0.44, meaning 22 of its top 50 ranked records were declining. Logistic Regression and Random Forest both achieved Precision@50 of 1.00, with all 50 of their top-ranked records observed as declining in this holdout.

For the full holdout classification task, Logistic Regression had an observed error rate of 26.14%, while Random Forest had an observed error rate of 19.36%.

The learned models therefore showed stronger observed top-50 prioritization than the ML-07 baseline on this client-held-out sample. Random Forest also had the lower overall classification error rate, while Logistic Regression provided a more directly interpretable coefficient-based view of the signals used by the model.

These results are directional and should be treated as decision-support evidence rather than proof of future performance. Further client-held-out validation would be needed before relying on the models in production.

### Final interpretation

On the client-held-out evaluation set, the ML-07 baseline achieved a measured Precision@50 of 0.44, while both Logistic Regression and Random Forest achieved 1.00.

This means the top 50 rankings from both learned models contained 50 observed declining pages, compared with 22 declining pages in the baseline's top 50.

The models should not be described as perfect. Precision@50 only evaluates the highest-ranked 50 records, and the result comes from one client-held-out evaluation split.

The broader error analysis and feature interpretation provide additional context for understanding where the models succeed and fail. The strongest measured features should be treated as predictive signals rather than causal explanations.

Overall, the ML models show a strong observed improvement over the ML-07 heuristic for this holdout ranking task. The result supports using the models as potential decision-support tools, subject to further validation on additional client-held-out samples.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.